# Volatility Dynamics 10d — wings priced from the matching volatility regime × time to expiry

**VD 028** (proposed in VD 024 §5). 10c priced every wing quote from one pooled two-year outcome distribution and found the bid CONSISTENT. That pool mixes regimes: standardized tails are about 10× fatter in calm volatility than in volatile volatility.

10d prices each quote from its own **(own-asset volatility tercile × TTE bucket)** cell of the two-year pool. The pool, the consistency gate and the Module 10 clock are the same as in 10c.

**Two pre-registered decisions** (Holm, family α = 0.05, one-sided; joint bootstrap of quote sessions × history dates, B = 1000):

| Decision | Sample | Price | Rejection means |
|---|---|---|---|
| D_long | low tercile (calm) | ask + fee | `CHEAP_AT_ASK`: a long-wing edge |
| D_short | high tercile (volatile) | bid − fee | `RICH_AT_BID`: a short-wing edge |

**Consequence stated in advance:** a rejection opens a defined-risk backtest pre-registration for that side only. No rejection closes the wing line.

Environment: `VD10D_HISTORY_DIR`, `VD10D_ASSETS`, `VD10D_N_BOOT`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_jump_option_wings as ow
from data_engineering.modeling import volatility_wings_empirical as we
from data_engineering.modeling import volatility_wings_longhistory as wl
from data_engineering.modeling import volatility_wings_regime as wr
ASSETS = [x.strip().upper() for x in os.getenv('VD10D_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
HIST_DIR = Path(os.getenv('VD10D_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
N_BOOT = int(os.getenv('VD10D_N_BOOT', str(wr.N_BOOT)))
VD = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics'
OUT = VD / '10d'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10d', wr.VERSION, wr.selftest(), '| 10c', wl.VERSION)
print(json.dumps(wr.registration(), indent=2))
manifest = HIST_DIR / 'manifest.json'
if manifest.exists():
    print(manifest.read_text())


## 1. Two-year pool and consistency gate (as 10c)

In [ ]:
t0 = time.time()
hist = wl.load_history(HIST_DIR, ASSETS)
print(hist.groupby('underlying').agg(first=('state_time', 'min'), last=('state_time', 'max'), n=('close', 'size'),
                                     source=('source', lambda s: ','.join(sorted(set(map(str, s)))))))
frame = wl.return_frame(hist)
from data_engineering.modeling import volatility_vd_b_core as vb
con = duckdb.connect(str(DB_PATH), read_only=True)
dbf = con.execute("""SELECT state_time, underlying, return_5m FROM time_series_feature_state
                     WHERE return_5m IS NOT NULL""").df()
con.close()
dbf = dbf[dbf.underlying.isin(ASSETS) & (pd.to_datetime(dbf.state_time, utc=True) < ow.VRP_CONFIRMATION_START)]
clock = wl.module10_clock(vb.build_grid(dbf))
oc = wl.long_outcomes(frame, clock=clock)
print(f'{len(oc):,} long-history origins in {time.time()-t0:.0f}s')
q = pd.read_csv(VD / '10' / '10_priced_quotes.csv.gz')
o10b = pd.read_csv(VD / '10b' / '10b_origin_outcomes.csv.gz')
pool_all, overlap = wl.split(oc, sorted(q.session.unique()))
gate = wl.consistency_gate(overlap, o10b)
display(gate.round(3))
failed = gate.loc[~gate['pass'], 'underlying'].tolist()
gate_fail = len(failed) > wl.GATE_MAX_FAILS
pool = pool_all[~pool_all.underlying.isin(failed)].reset_index(drop=True)
print('gate failures:', failed, '| GATE_FAIL:', gate_fail)
print(f'pool: {len(pool):,} origins, {pool.session.nunique()} dates, {pool.session.min()} -> {pool.session.max()}')


## 2. Volatility terciles for pool and quotes

In [ ]:
cuts = wr.cutoffs(pool)
pool['tercile'] = wr.classify(pool, cuts)
excluded = json.loads((VD / '10' / '10_decision.json').read_text()).get('excluded_assets_tick_artifact', [])
prim = ow.primary_sample(q, excluded).reset_index(drop=True)
prim = wr.attach_quote_v(prim, overlap)
prim = prim[np.isfinite(prim.v) & ~prim.underlying.isin(failed)].reset_index(drop=True)
prim['tercile'] = wr.classify(prim, cuts)
prim = wr.net_prices(prim)
display(pd.DataFrame(cuts, index=['cut_1/3', 'cut_2/3']).T)
display(prim.groupby(['tercile', 'tte_bucket']).size().unstack(fill_value=0))
display(pool.groupby(['tercile', 'tte_bucket']).size().unstack(fill_value=0))

## 3. Decisions

In [ ]:
t0 = time.time()
dec = wr.decide(prim, pool, n_boot=N_BOOT)
print(f'joint bootstrap ({time.time()-t0:.0f}s)')
print(json.dumps(dec, indent=2, default=float))

## 4. Reported: every cell at bid (net), mid and ask (net)

In [ ]:
cells = wr.cell_table(prim, pool, n_boot=500)
display(cells.pivot_table(index=['tercile', 'tte_bucket'], columns='price', values='M').round(3))

## 5. Outputs

In [ ]:
gate.to_csv(OUT / '10d_consistency_gate.csv', index=False)
cells.to_csv(OUT / '10d_cell_table.csv', index=False, float_format='%.6g')
pd.DataFrame(cuts, index=['cut_1', 'cut_2']).T.to_csv(OUT / '10d_tercile_cutoffs.csv')
(OUT / '10d_decision.json').write_text(json.dumps({'decision': dec, 'gate_failures': failed, 'gate_fail': gate_fail,
                                                    'registration': wr.registration()}, indent=2, default=float))
meta = {'version': wr.VERSION, 'module': '10d_regime_conditioned_wings', 'history_dir': str(HIST_DIR), 'n_boot': N_BOOT,
        'n_quotes': int(len(prim)), 'n_pool_origins': int(len(pool)), 'pool_last_session': wl.POOL_LAST_SESSION,
        'consumes': ['10/10_priced_quotes.csv.gz', '10b/10b_origin_outcomes.csv.gz', '10/10_decision.json']}
(OUT / '10d_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(meta, indent=2, default=float))